# Face Analysis Pipeline - Phase 4
## Age Classification using InsightFace (Pretrained)

**Approach:** Use InsightFace built-in age estimation (no fine-tuning needed)

**Input:** Aligned face crops from Phase 3
**Output:** Adult/Child classification + Confusion Matrix

## Step 1: Install Dependencies

In [14]:
!pip install scikit-learn seaborn -q
!pip install transformers torch pillow -q
print("✓ Dependencies installed")

✓ Dependencies installed


## Step 2: Mount Google Drive

In [15]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted


## Step 3: Import Libraries

In [16]:
import cv2
import json
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from insightface.app import FaceAnalysis
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


## Step 4: Configuration

In [17]:
BASE_PATH = Path("/content/drive/MyDrive/face_pipeline_project")
FACE_CROPS_FOLDER = BASE_PATH / "face_crops_aligned"
CLASSIFIED_FOLDER = BASE_PATH / "classified"
ADULT_FOLDER = CLASSIFIED_FOLDER / "adult"
CHILD_FOLDER = CLASSIFIED_FOLDER / "child"
METRICS_FOLDER = BASE_PATH / "metrics"
GROUND_TRUTH_FOLDER = BASE_PATH / "ground_truth"
CHECKPOINT_FILE = BASE_PATH / "checkpoint_age.json"

AGE_THRESHOLD = 18  # < 18 = Child, >= 18 = Adult

# Create folders
ADULT_FOLDER.mkdir(exist_ok=True, parents=True)
CHILD_FOLDER.mkdir(exist_ok=True, parents=True)
METRICS_FOLDER.mkdir(exist_ok=True, parents=True)

print(f"Input: {FACE_CROPS_FOLDER}")
print(f"Adult output: {ADULT_FOLDER}")
print(f"Child output: {CHILD_FOLDER}")
print(f"Age threshold: {AGE_THRESHOLD}")

Input: /content/drive/MyDrive/face_pipeline_project/face_crops_aligned
Adult output: /content/drive/MyDrive/face_pipeline_project/classified/adult
Child output: /content/drive/MyDrive/face_pipeline_project/classified/child
Age threshold: 18


## Step 5: Checkpoint Functions

In [18]:
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            cp = json.load(f)
            print(f"✓ Loaded: {len(cp.get('completed_videos', []))} videos done")
            return cp
    return {'completed_videos': [], 'total_adult': 0, 'total_child': 0, 'total_failed': 0}

def save_checkpoint(cp):
    cp['last_updated'] = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(cp, f, indent=2)

checkpoint = load_checkpoint()

## Step 6: Load InsightFace Model

In [19]:
from transformers import AutoImageProcessor, SiglipForImageClassification
from PIL import Image
import torch

print("Loading Prithiv facial-age-detection model...")
model_name = "prithivMLmods/facial-age-detection"
model = SiglipForImageClassification.from_pretrained(model_name)
processor = AutoImageProcessor.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Age class mapping
id2label = {
    0: "age 01-10",
    1: "age 11-20",
    2: "age 21-30",
    3: "age 31-40",
    4: "age 41-55",
    5: "age 56-65",
    6: "age 66-80",
    7: "age 80+"
}

# Child classes: 0, 1 (ages 1-20)
CHILD_CLASSES = {0, 1}

print(f"✓ Model loaded on {device}")
print(f"Child classes: {[id2label[i] for i in CHILD_CLASSES]}")

Loading Prithiv facial-age-detection model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✓ Model loaded on cpu
Child classes: ['age 01-10', 'age 11-20']


## Step 7: Process Face Crops - Age Classification

In [20]:
classifications_file = METRICS_FOLDER / "age_classifications.csv"
if classifications_file.exists():
    all_classifications = pd.read_csv(classifications_file).to_dict('records')
else:
    all_classifications = []

video_folders = sorted([f for f in FACE_CROPS_FOLDER.iterdir() if f.is_dir()])
completed = set(checkpoint['completed_videos'])
remaining = [v for v in video_folders if v.name not in completed]

print(f"Total: {len(video_folders)} | Completed: {len(completed)} | Remaining: {len(remaining)}")

for video_folder in remaining:
    video_name = video_folder.name
    print(f"\n{'='*50}")
    print(f"Processing: {video_name}")
    print(f"{'='*50}")

    (ADULT_FOLDER / video_name).mkdir(exist_ok=True)
    (CHILD_FOLDER / video_name).mkdir(exist_ok=True)

    face_files = sorted(list(video_folder.glob("*.jpg")))
    print(f"Faces: {len(face_files)}")

    adult_count = 0
    child_count = 0

    for face_path in tqdm(face_files, desc="Classifying"):
        try:
            # Load image
            img = Image.open(face_path).convert("RGB")

            # Preprocess
            inputs = processor(images=img, return_tensors="pt").to(device)

            # Predict
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits
                predicted_class = logits.argmax(dim=1).item()
                probs = torch.nn.functional.softmax(logits, dim=1).squeeze()
                confidence = probs[predicted_class].item()

            # Get age label
            age_label = id2label[predicted_class]

            # Classify as Adult or Child
            if predicted_class in CHILD_CLASSES:
                classification = 'child'
                dest_folder = CHILD_FOLDER / video_name
                child_count += 1
            else:
                classification = 'adult'
                dest_folder = ADULT_FOLDER / video_name
                adult_count += 1

            # Copy file
            shutil.copy(face_path, dest_folder / face_path.name)

            # Log
            all_classifications.append({
                'video': video_name,
                'face_file': face_path.name,
                'predicted_class': predicted_class,
                'age_label': age_label,
                'confidence': round(confidence, 3),
                'classification': classification,
                'is_adult': 1 if classification == 'adult' else 0
            })

        except Exception as e:
            print(f"Error: {face_path.name} - {e}")
            continue

    checkpoint['completed_videos'].append(video_name)
    checkpoint['total_adult'] += adult_count
    checkpoint['total_child'] += child_count
    save_checkpoint(checkpoint)

    print(f"✓ Adults: {adult_count} | Children: {child_count}")
    pd.DataFrame(all_classifications).to_csv(classifications_file, index=False)

print(f"\n✅ COMPLETE!")
print(f"Total Adults: {checkpoint['total_adult']}")
print(f"Total Children: {checkpoint['total_child']}")

Total: 27 | Completed: 0 | Remaining: 27

Processing: 09.14.00-10.18.46[M][0@0][126501]_ch1
Faces: 617


Classifying: 100%|██████████| 617/617 [10:44<00:00,  1.05s/it]


✓ Adults: 581 | Children: 36

Processing: 09.21.03-09.33.00[M][0@0][64136]_ch1
Faces: 18


Classifying: 100%|██████████| 18/18 [00:16<00:00,  1.11it/s]


✓ Adults: 7 | Children: 11

Processing: 09.22.21-09.37.43[M][0@0][54476]_ch1
Faces: 136


Classifying: 100%|██████████| 136/136 [02:03<00:00,  1.11it/s]


✓ Adults: 129 | Children: 7

Processing: 09.45.02-10.09.54[M][0@0][74131]_ch1
Faces: 180


Classifying:  56%|█████▌    | 100/180 [01:35<01:16,  1.05it/s]


KeyboardInterrupt: 

## Step 8: View Age Distribution

In [ ]:
df = pd.read_csv(METRICS_FOLDER / "age_classifications.csv")

print(f"Total classified: {len(df)}")
print(f"\nClassification breakdown:")
print(df['classification'].value_counts())

print(f"\nAge statistics:")
print(df['predicted_age'].describe())

# Histogram
plt.figure(figsize=(10, 5))
plt.hist(df['predicted_age'], bins=30, edgecolor='black', alpha=0.7)
plt.axvline(x=AGE_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold ({AGE_THRESHOLD})')
plt.xlabel('Predicted Age')
plt.ylabel('Count')
plt.title('Age Distribution (InsightFace)')
plt.legend()
plt.savefig(METRICS_FOLDER / "age_distribution.png", dpi=150)
plt.show()

---
# 📊 VALIDATION

## Step 9: Interactive Labeling Tool

In [ ]:
from IPython.display import display, clear_output, Image as IPImage
import ipywidgets as widgets

df_class = pd.read_csv(METRICS_FOLDER / "age_classifications.csv")

# Sample for labeling
SAMPLE_SIZE = 200
df_sample = df_class.sample(n=min(SAMPLE_SIZE, len(df_class)), random_state=42).reset_index(drop=True)

print(f"Total: {len(df_class)} | Sample: {len(df_sample)}")

# Labels storage
LABELS_FILE = GROUND_TRUTH_FOLDER / "age_labels_manual.csv"
labels = {}
current_idx = [0]

def save_labels():
    df_sample['actual_class'] = df_sample.index.map(lambda x: labels.get(x, ''))
    df_sample.to_csv(LABELS_FILE, index=False)
    return len([v for v in labels.values() if v != ''])

def show_face(idx):
    clear_output(wait=True)

    if idx >= len(df_sample):
        total = save_labels()
        print("✅ DONE!")
        print(f"Labeled: {total}")
        return

    row = df_sample.iloc[idx]
    img_path = FACE_CROPS_FOLDER / row['video'] / row['face_file']

    labeled_count = len([v for v in labels.values() if v != ''])

    print(f"Image {idx + 1}/{len(df_sample)}  |  Labeled: {labeled_count}")
    print(f"Model says: {row['classification'].upper()} (age: {row['predicted_age']})")

    if img_path.exists():
        display(IPImage(filename=str(img_path)))
    else:
        print(f"❌ Not found: {img_path}")

    btn_adult = widgets.Button(description="👨 ADULT", button_style='info', layout=widgets.Layout(width='150px', height='50px'))
    btn_child = widgets.Button(description="👶 CHILD", button_style='warning', layout=widgets.Layout(width='150px', height='50px'))
    btn_skip = widgets.Button(description="Skip", layout=widgets.Layout(width='100px'))
    btn_back = widgets.Button(description="← Back", layout=widgets.Layout(width='100px'))
    btn_save = widgets.Button(description="💾 Save", button_style='success', layout=widgets.Layout(width='100px'))

    def on_adult(b):
        labels[idx] = 1
        current_idx[0] += 1
        show_face(current_idx[0])

    def on_child(b):
        labels[idx] = 0
        current_idx[0] += 1
        show_face(current_idx[0])

    def on_skip(b):
        current_idx[0] += 1
        show_face(current_idx[0])

    def on_back(b):
        if current_idx[0] > 0:
            current_idx[0] -= 1
            show_face(current_idx[0])

    def on_save(b):
        total = save_labels()
        print(f"💾 Saved {total} labels")

    btn_adult.on_click(on_adult)
    btn_child.on_click(on_child)
    btn_skip.on_click(on_skip)
    btn_back.on_click(on_back)
    btn_save.on_click(on_save)

    display(widgets.HBox([btn_adult, btn_child]))
    display(widgets.HBox([btn_skip, btn_back, btn_save]))

print("🏁 Starting...")
show_face(0)

## Step 10: Confusion Matrix

In [ ]:
LABELS_FILE = GROUND_TRUTH_FOLDER / "age_labels_manual.csv"

try:
    df_gt = pd.read_csv(LABELS_FILE)
    df_labeled = df_gt[df_gt['actual_class'].notna() & (df_gt['actual_class'] != '')].copy()

    if len(df_labeled) == 0:
        print("⚠️ No labels! Run Step 9 first.")
    else:
        print(f"Labeled: {len(df_labeled)}")

        df_labeled['actual_class'] = df_labeled['actual_class'].astype(int)

        y_true = df_labeled['actual_class'].values
        y_pred = df_labeled['predicted_class'].values

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
                    xticklabels=['Child', 'Adult'],
                    yticklabels=['Child', 'Adult'])
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.title('Age Classification Confusion Matrix (InsightFace)')
        plt.savefig(METRICS_FOLDER / "confusion_matrix_age.png", dpi=150)
        plt.show()

        print("\n" + "="*50)
        print(classification_report(y_true, y_pred, labels=[0, 1], target_names=['Child', 'Adult'], zero_division=0))

        # Accuracy
        accuracy = (y_true == y_pred).mean()
        print(f"\nOverall Accuracy: {accuracy:.1%}")

        if accuracy >= 0.7:
            print("\n✅ InsightFace is working well! No fine-tuning needed.")
        else:
            print("\n⚠️ Accuracy below 70%. Consider fine-tuning.")

except FileNotFoundError:
    print("⚠️ Run Step 9 first!")

## Step 11: Display Samples

In [ ]:
adult_samples = list(ADULT_FOLDER.rglob("*.jpg"))[:4]
child_samples = list(CHILD_FOLDER.rglob("*.jpg"))[:4]

fig, axes = plt.subplots(2, 4, figsize=(14, 8))

for i, path in enumerate(adult_samples[:4]):
    if i < len(adult_samples):
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(img)
        axes[0, i].set_title("ADULT", fontsize=12, color='blue')
    axes[0, i].axis('off')

for i, path in enumerate(child_samples[:4]):
    if i < len(child_samples):
        img = cv2.imread(str(path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[1, i].imshow(img)
        axes[1, i].set_title("CHILD", fontsize=12, color='orange')
    axes[1, i].axis('off')

plt.suptitle("Sample Classifications", fontsize=14)
plt.tight_layout()
plt.savefig(METRICS_FOLDER / "sample_classifications.png", dpi=150)
plt.show()

---
## ✅ Phase 4 Complete!

**Results:**
- Adult faces: `/classified/adult/`
- Child faces: `/classified/child/`
- Age log: `/metrics/age_classifications.csv`
- Confusion matrix: `/metrics/confusion_matrix_age.png`

**Next: Phase 5 - Emotion Classification (Child faces only)**